# Rule-based candidate selection over saved Notebook 17 candidates (CPU)

**Question.** On dev, a correct program is among the 4 beams for 713 of 883 questions, but the top
beam is correct for 625. Do four fixed rules (`finevidence.generation.rule_selection`) recover part
of that gap without any new decoding?

**Rules, fixed before evaluation.** Reject a candidate that is malformed, fails controlled execution
(including references to a later or missing step), uses a numeric literal absent from the retrieved
facts, or aggregates a table row whose label is not in the retrieved facts. Choose the surviving
candidate with the highest length-normalized log-probability; keep the top beam if none survives.
Nothing is fitted, so there is nothing to tune on dev.

**Protocol.** Two pools: the 4 beams, and all 20 candidates (4 beams plus 16 samples). Predictions
are made from candidates, retrieved facts and raw tables only, written to disk and hashed **before**
any dev label is read. Then they are scored and compared with the top beam (625/883) by an exact
two-sided McNemar test. Saved inputs are read, never modified. Test is not loaded. No GPU.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest

from finevidence.data import load_finqa_split
from finevidence.generation import load_rankings
from finevidence.generation.inputs import build_keyed_example
from finevidence.generation.rule_selection import REJECTION_REASONS, select_by_rules

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
processed = PROJECT_ROOT / "data" / "processed"
EXPECTED = {
    "generator-qwen2.5-1.5b-lora-top10-v1_dev_candidates.jsonl": (
        "237557f60207ee8876a506525791a0db3926f9f4ad736125ae4f89ef7fd38c92"
    ),
    "generator-qwen2.5-1.5b-lora-top10-v1_dev_retrieved_top_beam_predictions.jsonl": (
        "6a8b8f9d8f165bc14b19d360ac6720f05aa877f9dbd4c31b2b705426f84ec418"
    ),
    "retriever_bert_base_dev_rankings.jsonl": (
        "d6c31f86c1fd4ab0d68d2cb56996ffe9705fba7dfbfe62464fd7b0ea7bd562e6"
    ),
}
for name, sha in EXPECTED.items():
    assert hashlib.sha256((processed / name).read_bytes()).hexdigest() == sha, name
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
print({"commit": commit, "python": platform.python_version(), "inputs_verified": True})

{'commit': '515d6ff90f9cfcdfd531056453a454e9f9d81ea5', 'python': '3.13.16', 'inputs_verified': True}


## 1. Audit: alignment and provenance of the saved candidates

In [2]:
dev = load_finqa_split(PROJECT_ROOT / "data" / "raw" / "dev.json")
rankings = load_rankings(processed / "retriever_bert_base_dev_rankings.jsonl")
candidates = [
    json.loads(line)
    for line in (processed / "generator-qwen2.5-1.5b-lora-top10-v1_dev_candidates.jsonl")
    .read_text()
    .splitlines()
]
baseline = [
    json.loads(line)
    for line in (
        processed / "generator-qwen2.5-1.5b-lora-top10-v1_dev_retrieved_top_beam_predictions.jsonl"
    )
    .read_text()
    .splitlines()
]
assert [row["record_id"] for row in candidates] == [str(record["id"]) for record in dev]
assert [row["record_id"] for row in baseline] == [row["record_id"] for row in candidates]
assert {row["commit"] for row in candidates} == {row["commit"] for row in baseline}
assert all(row["fact_keys"] == rankings[row["record_id"]][:10] for row in candidates)
assert all(
    c["beam_candidates"][0] == b["prediction"] for c, b in zip(candidates, baseline, strict=True)
)
assert all(
    len(row["beam_candidates"]) == len(row["beam_scores"]) == 4
    and len(row["sample_candidates"]) == len(row["sample_scores"]) == 16
    for row in candidates
)
print(
    {
        "records": len(candidates),
        "generator_commit": candidates[0]["commit"],
        "fact_keys_are_ranking_top10": True,
        "beam0_is_baseline": True,
    }
)

{'records': 883, 'generator_commit': '6cc650e002f1851f1801346ad9d17289fb080282', 'fact_keys_are_ranking_top10': True, 'beam0_is_baseline': True}


## 2. Predictions (no labels read)

Fact texts are rebuilt from each record's raw context by the saved fact keys
(`build_keyed_example`, which reads the question and context, not `qa` annotations). Both pools'
predictions are written and hashed before scoring.

In [3]:
pools = {
    "beams4": lambda row: (row["beam_candidates"], row["beam_scores"]),
    "all20": lambda row: (
        row["beam_candidates"] + row["sample_candidates"],
        row["beam_scores"] + row["sample_scores"],
    ),
}
label_free = [
    {
        "id": record["id"],
        "table": record["table"],
        "pre_text": record["pre_text"],
        "post_text": record["post_text"],
        "qa": {"question": record["qa"]["question"]},
    }
    for record in dev
]
fact_texts = [
    build_keyed_example(record, row["fact_keys"], None).fact_texts
    for record, row in zip(label_free, candidates, strict=True)
]
selections, timing, frozen = {}, {}, {}
for pool, take in pools.items():
    started = time.perf_counter()
    selections[pool] = [
        select_by_rules(*take(row), facts, record["table"])
        for row, facts, record in zip(candidates, fact_texts, label_free, strict=True)
    ]
    timing[pool] = (time.perf_counter() - started) / len(candidates)
    path = processed / f"rule-selection_{pool}_dev_predictions.jsonl"
    with path.open("w", encoding="utf-8") as handle:
        for row, chosen in zip(candidates, selections[pool], strict=True):
            program = take(row)[0][chosen.index]
            handle.write(
                json.dumps(
                    {
                        "record_id": row["record_id"],
                        "prediction": program,
                        "index": chosen.index,
                        "fallback": chosen.fallback,
                        "rejections": chosen.rejections,
                        "commit": commit,
                    }
                )
                + "\n"
            )
    frozen[pool] = hashlib.sha256(path.read_bytes()).hexdigest()
print(
    json.dumps(
        {
            "frozen_sha256": frozen,
            "cpu_ms_per_question": {k: round(v * 1000, 2) for k, v in timing.items()},
        },
        indent=2,
    )
)

{
  "frozen_sha256": {
    "beams4": "ecf152cabdacc6f502cf34c07f6e8f50ccf46ba2244d4dd24cf29b707e397e5c",
    "all20": "e986bf9e3102c598a9e17ab587e43558c4f6c364695b8baa8c54bfca2bb7e50a"
  },
  "cpu_ms_per_question": {
    "beams4": 0.42,
    "all20": 1.96
  }
}


## 3. Rejections by reason

In [4]:
rows = []
for pool, chosen_list in selections.items():
    counts = Counter(r for chosen in chosen_list for r in chosen.rejections)
    rows.append(
        {
            "pool": pool,
            "candidates": sum(len(c.rejections) for c in chosen_list),
            **{reason: counts.get(reason, 0) for reason in REJECTION_REASONS},
            "survived": counts.get(None, 0),
            "questions_with_no_survivor": sum(c.fallback for c in chosen_list),
            "prediction_changed": sum(
                " ".join(str(pools[pool](row)[0][c.index]).split())
                != " ".join(row["beam_candidates"][0].split())
                for c, row in zip(chosen_list, candidates, strict=True)
            ),
        }
    )
display(pd.DataFrame(rows).set_index("pool"))

,candidates,malformed,execution,ungrounded_literal,table_row_not_in_facts,survived,questions_with_no_survivor,prediction_changed
pool,,,,,,,,
beams4,3532,17,11,36,0,3468,4,1
all20,17660,24,17,119,0,17500,1,82


## 4. Scoring against the top beam (labels read only here, after predictions were frozen)

In [5]:
from finevidence.generation import score_program_prediction  # noqa: E402

old = np.array([row["execution_correct"] for row in baseline])
assert old.sum() == 625
results = {}
for pool in pools:
    path = processed / f"rule-selection_{pool}_dev_predictions.jsonl"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == frozen[pool]
    predicted = [json.loads(line)["prediction"] for line in path.read_text().splitlines()]
    scores = [
        score_program_prediction(record, program, facts)
        for record, program, facts in zip(dev, predicted, fact_texts, strict=True)
    ]
    new = np.array([s.execution_correct for s in scores])
    changed = [
        " ".join(str(p).split()) != " ".join(b["prediction"].split())
        for p, b in zip(predicted, baseline, strict=True)
    ]
    fixed, broken = int((new & ~old).sum()), int((old & ~new).sum())
    results[pool] = {
        "execution_correct": int(new.sum()),
        "execution_accuracy": round(float(new.mean()), 4),
        "program_accuracy": round(float(np.mean([s.program_correct for s in scores])), 4),
        "changed": int(sum(changed)),
        "fixed": fixed,
        "broken": broken,
        "net": fixed - broken,
        "mcnemar_exact_p": round(float(binomtest(fixed, fixed + broken).pvalue), 6)
        if fixed + broken
        else 1.0,
        "cpu_ms_per_question": round(timing[pool] * 1000, 2),
    }
display(pd.DataFrame(results).T)

,execution_correct,execution_accuracy,program_accuracy,changed,fixed,broken,net,mcnemar_exact_p,cpu_ms_per_question
beams4,625.0,0.7078,0.6636,1.0,0.0,0.0,0.0,1.000000,0.42
all20,613.0,0.6942,0.6489,82.0,5.0,17.0,-12.0,0.016901,1.96
